# HW3 Part 2


**Course Code:** 515513

**Group number:** 515513_07

**Student Name:** 呂泰廷 吳瑞傑 郭宗睿

**Student ID:** 112652030 112652027 111652043

# Turn Continuity Classification

**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |

## 0. Environment Setup & Data Loading

In [2]:
!pip install datasets xgboost imbalanced-learn nltk -q


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Core ML
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix

# Advanced
import xgboost as xgb
from imblearn.over_sampling import SMOTE

# NLP
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('omw-1.4', quiet=True)


print("All imports successful!")

All imports successful!


In [4]:
# Load dataset (ensure train.csv and test.csv are in your current directory)
train_path = 'train.csv'
test_path = 'test.csv'

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

# Basic exploration
print(f"Train size: {len(train_df)}")
print("Label Distribution:\n", train_df['label'].value_counts())
display(train_df.head())

Train size: 1302
Label Distribution:
 label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [5]:
# ── Standardize column names ──────────────────────────────────────

# Based on your files, the text column is named 'content'
# We will normalize it to 'text' for consistency in the pipeline
text_col_mapping = {'content': 'text'}

# Rename 'content' to 'text' if it exists
train_df = train_df.rename(columns=text_col_mapping)
public_test_df = public_test_df.rename(columns=text_col_mapping)

# Ensure 'id' column exists (if not already present)
if 'id' not in train_df.columns:
    train_df['id'] = train_df.index
if 'id' not in public_test_df.columns:
    public_test_df['id'] = public_test_df.index

# Ensure text columns are strings
train_df['text'] = train_df['text'].fillna('').astype(str)
public_test_df['text'] = public_test_df['text'].fillna('').astype(str)

# ── Label Analysis (Training Set Only) ────────────────────────────

print("Standardization complete.")
print(f"Final columns: {train_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print("\nLabel distribution (train):")
    print(counts)

    if 0 in counts and 1 in counts:
        ratio = counts[0] / counts[1]
        print(f"\nImbalance ratio (0/1): {ratio:.2f}")
    else:
        print("\nNote: Only one class found in 'label' column.")
else:
    print("\nWarning: 'label' column not found in train_df.")

Standardization complete.
Final columns: ['id', 'text', 'label']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64

Imbalance ratio (0/1): 0.56


## Step 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

We choose **Cost-Sensitive Learning** as the advanced method. It re-weights the loss function instead of duplicating rows, which mathematically achieves balanced training without introducing duplicate samples that can mildly over-fit.

In [ ]:
# =================================================================
# Step 1 - Data balancing
# =================================================================
# Goal: compare two imbalance-handling methods under the same
# TF-IDF + Linear SVM setup.
# Method A: Random Over-sampling duplicates minority-class rows
# until labels 0 and 1 have the same count.
# Method B: Cost-Sensitive Learning keeps the data unchanged and
# later uses class_weight='balanced' in the SVM.

def perform_balancing(df, method='random'):
    if method == 'random':
        # Split rows by class. Label 1 is the majority class,
        # and label 0 is the minority class in this dataset.
        df_majority = df[df['label'] == 1]
        df_minority = df[df['label'] == 0]

        # Upsample minority rows with replacement until both classes match.
        df_minority_upsampled = resample(
            df_minority,
            replace=True,                   # sample with replacement
            n_samples=len(df_majority),     # match the majority-class count
            random_state=42,                # keep sampling reproducible
        )

        # Merge both classes and shuffle so duplicated rows are mixed in.
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    elif method == 'advanced':
        # Cost-Sensitive Learning does not duplicate any row.
        # The SVM receives class_weight='balanced' during training,
        # which makes mistakes on the minority class more expensive.
        return df.copy().reset_index(drop=True)

# Quick check: confirm both balancing paths produce the expected label counts.
sample_balanced = perform_balancing(train_df, method='random')
print("After Random Over-sampling (兩類 1:1):")
print(sample_balanced['label'].value_counts())

sample_adv = perform_balancing(train_df, method='advanced')
print("\nAfter Cost-Sensitive (資料不變，靠 class_weight 平衡):")
print(sample_adv['label'].value_counts())


## Steps 2-6: Preprocessing, Baseline, and Balancing Comparison

Prepare endpoint-aware text features, define the models, run the baseline, and compare balancing methods using Macro-F1.

In [ ]:
# =================================================================
# Step 2 - Endpoint-aware text preprocessing
# =================================================================
# Key idea: incomplete turns often stop at a function word, such as
# the, to, of, could, or if. Standard TF-IDF sees these words, but it
# does not know whether they appear at the end of the utterance.
# We append explicit ending-marker tokens so the SVM can learn
# endpoint patterns directly.

# Word groups used to describe the final token of an utterance.
FILLER = {'um','uh','er','like','maybe','actually','wait','well','okay','ok','so','just'}
DET    = {'the','a','an','my','your','his','her','its','our','their','this','that','these','those','some','any','no','every'}  # determiners
PREP   = {'of','to','in','on','at','with','for','from','about','into','over','under','through','during','before','after','between','as','by'}  # prepositions
CONJ   = {'and','but','or','because','if','when','while','though','although','since','unless','that','whether','so','than'}  # conjunctions
MODAL  = {'can','could','will','would','shall','should','may','might','must','do','does','did','have','has','had','is','was','were','am','are','be','been','being'}  # auxiliaries / be verbs
PRON   = {'i','you','he','she','it','we','they','me','him','us','them','myself','yourself','himself','herself','itself','ourselves','themselves'}  # pronouns

def _cat(w):
    """Map one token to a coarse endpoint category."""
    if w in FILLER: return 'FILLER'
    if w in DET:    return 'DET'
    if w in PREP:   return 'PREP'
    if w in CONJ:   return 'CONJ'
    if w in MODAL:  return 'AUX'
    if w in PRON:   return 'PRON'
    return 'CONTENT'

_TOKRE = re.compile(r"[A-Za-z']+")  # keep English words only

def preprocess_text(text):
    """Append endpoint markers so TF-IDF can learn final-token patterns."""
    if not isinstance(text, str):
        return ''
    t = text.lower().strip()
    toks = _TOKRE.findall(t)            # simple word tokenization
    if not toks:
        return t
    end1 = toks[-1]                                                     # last word
    end2 = '_'.join(toks[-2:])  if len(toks) >= 2 else end1             # last two words
    end3 = '_'.join(toks[-3:])  if len(toks) >= 3 else end2             # last three words
    cat1 = _cat(end1)                                                   # category of last word
    cat2 = _cat(toks[-2]) if len(toks) >= 2 else cat1                   # category of second-last word
    # Add five marker tokens: final words plus final-word categories.
    return f'{t} __END1_{end1}__ __END2_{end2}__ __END3_{end3}__ __CAT1_{cat1}__ __CAT12_{cat2}_{cat1}__'

# Apply the same preprocessing to train and test to avoid train/test mismatch.
train_df['text_clean']       = train_df['text'].apply(preprocess_text)
public_test_df['text_clean'] = public_test_df['text'].apply(preprocess_text)

# Print one example to verify the added endpoint markers.
print('原始 :', train_df['text'].iloc[0])
print('增強 :', train_df['text_clean'].iloc[0])


In [ ]:
# =================================================================
# Step 3 - Model factory
# =================================================================
# Keep model creation in one helper so all experiments use consistent settings.
#   'svm'      = Linear SVM baseline; class_weight controls cost sensitivity.
#   'advanced' = XGBoost reference model for comparison.
def get_model(model_type='svm', class_weight=None):
    if model_type == 'svm':
        # Linear SVM works well with sparse TF-IDF features.
        # class_weight='balanced' turns it into the cost-sensitive version.
        return SVC(kernel='linear', C=1.0, class_weight=class_weight,
                   probability=True, random_state=42)

    elif model_type == 'advanced':
        # XGBoost is a tree-based advanced-model reference.
        return xgb.XGBClassifier(
            n_estimators=400,        # number of trees
            max_depth=6,             # maximum depth per tree
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=42,
            n_jobs=-1,
        )


In [9]:
# Step 4 - Internal hold-out validation split
train_data, val_data = train_test_split(
    train_df, test_size=0.2, random_state=42, stratify=train_df['label'])  # keep label ratio stable


print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print("Train label distribution:")
print(train_data['label'].value_counts())

Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64


In [ ]:
# =================================================================
# Step 5 - Baseline: TF-IDF unigram + Linear SVM
# =================================================================
# Required baseline using Random Over-sampling.
# Process: balance training split -> fit TF-IDF -> train SVM -> evaluate Macro-F1.

print("Running Baseline Experiment...")

MODEL_TYPE     = 'svm'      # required Linear SVM baseline
BALANCE_METHOD = 'random'   # required Random Over-sampling

# 1. Balance the training split to a 1:1 label ratio.
balanced_train_data = perform_balancing(train_data.copy(), method=BALANCE_METHOD)
print(f"Balanced train size: {len(balanced_train_data)}  |  label dist:")
print(balanced_train_data['label'].value_counts())

# 2. Convert text to unigram TF-IDF features.
# ngram_range=(1,1) uses single words; min_df=2 drops very rare terms.
vectorizer = TfidfVectorizer(ngram_range=(1, 1), min_df=2, stop_words=None)
X_train = vectorizer.fit_transform(balanced_train_data['text_clean'])  # fit only on training data
y_train = balanced_train_data['label']

# 3. Train the SVM baseline.
clf = get_model(MODEL_TYPE)
clf.fit(X_train, y_train)
print(f"Baseline model ({MODEL_TYPE}) trained on {X_train.shape[0]} balanced samples.")

# 4. Evaluate Macro-F1 on the validation split.
X_val = vectorizer.transform(val_data['text_clean'])     # transform with the same vectorizer
y_val = val_data['label']
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')

print(f"=== Baseline: {MODEL_TYPE} | Balancing: {BALANCE_METHOD} ===")
print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
print(f"Macro-F1 Score: {macro_f1:.4f}")


In [11]:
# Step 6 - Compare balancing methods fairly
# Both methods use the same unigram TF-IDF + Linear SVM setup.
# The only intended difference is duplicated minority rows vs class_weight='balanced'.

results_bal = {}
for bal in ['random', 'advanced']:
    bal_tr = perform_balancing(train_data.copy(), method=bal)
    cw = 'balanced' if bal == 'advanced' else None
    v = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
    Xtr = v.fit_transform(bal_tr['text_clean'])
    Xva = v.transform(val_data['text_clean'])
    m = get_model('svm', class_weight=cw)
    m.fit(Xtr, bal_tr['label'])
    p = m.predict(Xva)
    score = f1_score(val_data['label'], p, average='macro')
    results_bal[bal] = score
    label = 'Random Over-sampling' if bal == 'random' else 'Cost-Sensitive (advanced)'
    print(f'{label:32s}  macro-F1 = {score:.4f}')

print('\nBest balancing method:', max(results_bal, key=results_bal.get))

Random Over-sampling              macro-F1 = 0.8565
Cost-Sensitive (advanced)         macro-F1 = 0.8774

Best balancing method: advanced


## Step 7: Stratified K-Fold Experiments

Use Stratified 5-Fold validation to compare the baseline, endpoint-aware SVM, and XGBoost reference more reliably than a single split.

In [ ]:
# =================================================================
# Step 7 - Stratified K-Fold validation
# =================================================================
# A single 80/20 split can be lucky or unlucky. Five folds give a
# more stable estimate by rotating which samples are used for validation.
# Stratified splitting keeps the label ratio similar in every fold,
# which is important for this imbalanced dataset.

def run_kfold_cv(df, model_type='svm', balance_method='random', n_splits=5,
                 use_custom_preprocessing=True, ngram_range=(1, 2),
                 sublinear_tf=True, min_df=2, C=1.0):
    """Run Stratified K-Fold CV and return the Macro-F1 score for each fold."""
    # use_custom_preprocessing=True means using endpoint-augmented text_clean.
    text_col = 'text_clean' if use_custom_preprocessing else 'text'
    # balance_method='advanced' uses class_weight='balanced' instead of duplicated rows.
    cw = 'balanced' if balance_method == 'advanced' else None

    X_all = df[text_col].values
    y_all = df['label'].values
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_scores = []
    for fold, (tri, vai) in enumerate(skf.split(X_all, y_all), 1):
        # Split this fold into train and validation parts.
        tr_df_fold = df.iloc[tri].reset_index(drop=True)
        va_df_fold = df.iloc[vai].reset_index(drop=True)

        # Only Random Over-sampling changes the data; advanced uses model weights.
        if balance_method == 'random':
            tr_df_fold = perform_balancing(tr_df_fold, method='random')

        # Fit TF-IDF only on the training fold to avoid validation vocabulary leakage.
        v = TfidfVectorizer(ngram_range=ngram_range, min_df=min_df,
                            sublinear_tf=sublinear_tf)
        Xtr = v.fit_transform(tr_df_fold[text_col])
        Xva = v.transform(va_df_fold[text_col])

        # Train, predict, and compute Macro-F1.
        if model_type == 'svm':
            m = SVC(kernel='linear', C=C, class_weight=cw, random_state=42)
        else:
            m = get_model('advanced')
        m.fit(Xtr, tr_df_fold['label'])
        p = m.predict(Xva)
        sc = f1_score(va_df_fold['label'], p, average='macro')
        fold_scores.append(sc)
        print(f'  fold {fold}/{n_splits}  macro-F1 = {sc:.4f}')
    print(f'>> mean = {np.mean(fold_scores):.4f}  std = {np.std(fold_scores):.4f}')
    return fold_scores

# Run three comparable settings.

# (1) Baseline: raw text + unigram TF-IDF + Random Over-sampling
print('--- Baseline: SVM unigram + random over-sampling (原始文字) ---')
base_scores = run_kfold_cv(train_df, model_type='svm', balance_method='random',
                           use_custom_preprocessing=False, ngram_range=(1, 1),
                           sublinear_tf=False)

# (2) Advanced SVM: endpoint markers + (1,3)-gram TF-IDF + cost sensitivity
print('\n--- Advanced: SVM (1,3)-gram + sublinear + cost-sensitive + endpoint augment ---')
adv_scores = run_kfold_cv(train_df, model_type='svm', balance_method='advanced',
                          use_custom_preprocessing=True, ngram_range=(1, 3),
                          sublinear_tf=True, C=0.5)

# (3) Reference model: XGBoost on the same TF-IDF-style features
print('\n--- Reference: XGBoost on TF-IDF (1,2)-gram + sublinear ---')
xgb_scores = run_kfold_cv(train_df, model_type='advanced', balance_method='random',
                          use_custom_preprocessing=True, ngram_range=(1, 2),
                          sublinear_tf=True)


## Steps 8-9: Final Training and Kaggle Submission

Train the selected model on the full training set and generate `submission.csv` for Kaggle.

In [13]:
# Step 8 - Final training on all labeled data
# After selecting the best CV configuration, retrain once on the full
# training set before predicting the Kaggle test set.
# Selected config: endpoint-aware text + TF-IDF (1,3)-gram +
# Linear SVM with C=0.5 and class_weight='balanced'.

BEST_BALANCE = 'advanced'        # use cost-sensitive learning
final_balanced_train_df = perform_balancing(train_df.copy(), method=BEST_BALANCE)
print(f"Full training size: {len(final_balanced_train_df)}")
print(final_balanced_train_df['label'].value_counts())

final_vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=2, sublinear_tf=True)
X_full_train = final_vectorizer.fit_transform(final_balanced_train_df['text_clean'])
y_full_train = final_balanced_train_df['label']

final_model = SVC(kernel='linear', C=0.5, class_weight='balanced', random_state=42)
final_model.fit(X_full_train, y_full_train)
print(f"Final model trained on {X_full_train.shape[0]} samples (augmented preprocess_text).")


Full training size: 1302
label
1    837
0    465
Name: count, dtype: int64
Final model trained on 1302 samples (augmented preprocess_text).


In [14]:
# -----------------------------------------------------------------
# Step 9 - Kaggle submission file
# -----------------------------------------------------------------

def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    """Create submission.csv with the required id,label columns."""
    text_col = 'text_clean' if 'text_clean' in test_df.columns else 'text'
    X_test = vectorizer.transform(test_df[text_col])
    test_predictions = model.predict(X_test).astype(int)

    submission = pd.DataFrame({'id': test_df['id'].astype(int), 'label': test_predictions})
    submission.to_csv(output_name, index=False)

    print(f"Saved → {output_name}")
    print("Prediction distribution:")
    print(submission['label'].value_counts())
    display(submission.head(10))

    print(f"Success: {output_name} ready for Kaggle upload.")

# Final Step:
generate_kaggle_submission(final_model, final_vectorizer, public_test_df)

Saved → submission.csv
Prediction distribution:
label
1    304
0    196
Name: count, dtype: int64


,id,label
0,1608,0
1,334,1
2,1779,1
3,1015,1
4,1790,1
5,1635,1
6,358,0
7,1574,1
8,700,1
9,284,1


Success: submission.csv ready for Kaggle upload.


In [15]:
# Optional: automatically download submission.csv when running in Google Colab.
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")

Not running in Colab — file saved locally as submission.csv
